# CV06 — Test-time adaptation

Source classifier đã train gặp ảnh bị corruption. TENT cố định trọng số convolution, chỉ cập nhật affine của BatchNorm bằng entropy. Chỉ áp dụng khi luật cho phép học từ input test.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Source classifier đã train gặp ảnh bị corruption. TENT cố định trọng số convolution, chỉ cập nhật affine của BatchNorm bằng entropy. Chỉ áp dụng khi luật cho phép học từ input test. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from copy import deepcopy
from torchvision.models import resnet18
from vision.curriculum.cv06_adapt import configure_tent, adapt, entropy
images = torch.rand(2, 3, 32, 32, device=device)
source_model = resnet18(weights=None, num_classes=2).to(device).eval()
model = deepcopy(source_model)
optimizer = configure_tent(model)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
with torch.no_grad():
    source_logits = source_model(images)
print('Source entropy:', entropy(source_logits).mean().item())
print('Trainable BN parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
# adapt computes entropy, backward and optimizer.step internally.
before = next(p for p in model.parameters() if p.requires_grad).detach().clone()
logits_before_update = adapt(model, optimizer, images)
after = next(p for p in model.parameters() if p.requires_grad).detach()
print('Adapted logits:', logits_before_update.shape)
print('BN affine change:', (after - before).norm().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Frozen source → BN statistics → TENT entropy minimization → entropy filtering (chưa phải toàn bộ EATA). Phải có source checkpoint đã học để đánh giá adaptation.


In [ ]:
model = deepcopy(source_model)
optimizer = configure_tent(model)
print('Reset to immutable source before the next domain.')
print('No labels are consumed by adapt; random source weights are only a mechanics demo.')


## Thử nghiệm tiếp theo

Frozen source → BN statistics → TENT entropy minimization → entropy filtering (chưa phải toàn bộ EATA). Phải có source checkpoint đã học để đánh giá adaptation.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv06_adapt.py) và [tài liệu gốc](../REFERENCE.md).
